# Confidence Intervals

## Learning Objectives
1. Construct z-CI and t-CI for the mean from first principles using numpy and scipy
2. Implement bootstrap CI by resampling and compare it to analytical CIs via simulation
3. Apply Wilson score interval for proportions and identify when the normal approximation breaks down
4. Quantify the relationship between sample size and CI width and compute the required n for a target margin of error

In [ ]:
import numpy as np
import scipy.stats as stats
import matplotlib.pyplot as plt
from typing import Tuple, Optional

np.random.seed(42)

# Confirm key imports work
print("numpy version:", np.__version__)
print("scipy version:", __import__('scipy').__version__)
print("Setup complete - seed set to 42")

## Level 1: Manual 95% CI for the Mean (z and t)

We build confidence intervals from first principles.
- **z-CI**: use when sigma known or n >= 30 (uses z* = 1.96 for 95%)
- **t-CI**: use when sigma unknown, especially for small n (uses t* with n-1 degrees of freedom)

The margin of error MOE = critical_value * SE, where SE = s / sqrt(n).

In [ ]:
# -----------------------------------------------------------------------
# Level 1: Manual CI for the mean using z and t distributions
# -----------------------------------------------------------------------

def z_confidence_interval(
    data: np.ndarray,
    confidence: float = 0.95,
    sigma: Optional[float] = None
) -> Tuple[float, float, float, float]:
    """
    Compute z-based confidence interval for the population mean.

    Parameters
    ----------
    data       : 1-D array of observations
    confidence : confidence level, e.g. 0.95
    sigma      : known population std; if None, uses sample std (large-n approximation)

    Returns
    -------
    (lower, upper, mean, margin_of_error)
    """
    n = len(data)
    mean = np.mean(data)
    se = (sigma if sigma is not None else np.std(data, ddof=1)) / np.sqrt(n)
    alpha = 1 - confidence
    z_star = stats.norm.ppf(1 - alpha / 2)  # two-tailed critical value
    moe = z_star * se
    return mean - moe, mean + moe, mean, moe


def t_confidence_interval(
    data: np.ndarray,
    confidence: float = 0.95
) -> Tuple[float, float, float, float]:
    """
    Compute t-based confidence interval for the population mean.

    Uses n-1 degrees of freedom, appropriate when sigma is unknown.
    For n < 30, t* is noticeably larger than z*, giving a wider (more honest) interval.

    Parameters
    ----------
    data       : 1-D array of observations
    confidence : confidence level

    Returns
    -------
    (lower, upper, mean, margin_of_error)
    """
    n = len(data)
    mean = np.mean(data)
    se = np.std(data, ddof=1) / np.sqrt(n)
    alpha = 1 - confidence
    t_star = stats.t.ppf(1 - alpha / 2, df=n - 1)  # t critical value
    moe = t_star * se
    return mean - moe, mean + moe, mean, moe


# ------ Demonstration ------
true_mean = 50.0
true_std = 10.0

# Small sample: n=15, t-CI should be noticeably wider than z-CI
small_sample = np.random.normal(true_mean, true_std, size=15)
z_lo_s, z_hi_s, mean_s, moe_z_s = z_confidence_interval(small_sample)
t_lo_s, t_hi_s, _, moe_t_s = t_confidence_interval(small_sample)

print(f"Small sample (n=15), sample mean = {mean_s:.2f}, true mean = {true_mean}")
print(f"  z-CI: [{z_lo_s:.2f}, {z_hi_s:.2f}]  width = {z_hi_s - z_lo_s:.2f}")
print(f"  t-CI: [{t_lo_s:.2f}, {t_hi_s:.2f}]  width = {t_hi_s - t_lo_s:.2f}")
print(f"  t-CI is {(moe_t_s / moe_z_s - 1) * 100:.1f}% wider than z-CI for n=15")

# Large sample: n=200, z and t should converge
large_sample = np.random.normal(true_mean, true_std, size=200)
z_lo_l, z_hi_l, mean_l, _ = z_confidence_interval(large_sample)
t_lo_l, t_hi_l, _, _ = t_confidence_interval(large_sample)

print(f"\nLarge sample (n=200), sample mean = {mean_l:.2f}")
print(f"  z-CI: [{z_lo_l:.2f}, {z_hi_l:.2f}]  width = {z_hi_l - z_lo_l:.2f}")
print(f"  t-CI: [{t_lo_l:.2f}, {t_hi_l:.2f}]  width = {t_hi_l - t_lo_l:.2f}")
print("  z and t converge for large n as expected")

# Critical value comparison across sample sizes
print("\nCritical values: z* vs t* by sample size")
print(f"{'n':>6}  {'z*':>6}  {'t*':>6}  {'% diff':>8}")
z_star_95 = stats.norm.ppf(0.975)
for n in [5, 10, 15, 20, 30, 50, 100, 200]:
    t_star = stats.t.ppf(0.975, df=n - 1)
    diff = (t_star / z_star_95 - 1) * 100
    print(f"{n:>6}  {z_star_95:>6.3f}  {t_star:>6.3f}  {diff:>7.1f}%")

## Level 2: Bootstrap CI

Bootstrap CI is **distribution-free**: it makes no assumption about the data distribution.
We resample the data with replacement 10,000 times, compute the statistic each time,
and take the 2.5th and 97.5th percentiles as the 95% CI bounds.

This works for any statistic: mean, median, correlation, AUC, etc.

In [ ]:
# -----------------------------------------------------------------------
# Level 2: Bootstrap CI + coverage probability simulation
# -----------------------------------------------------------------------

def bootstrap_ci(
    data: np.ndarray,
    statistic: callable = np.mean,
    n_boot: int = 10_000,
    confidence: float = 0.95,
    random_state: int = 42
) -> Tuple[float, float, np.ndarray]:
    """
    Compute bootstrap percentile confidence interval.

    Parameters
    ----------
    data         : 1-D observed data array
    statistic    : function mapping array -> scalar (e.g. np.mean, np.median)
    n_boot       : number of bootstrap resamples (10K for stable 95% CI)
    confidence   : confidence level
    random_state : RNG seed for reproducibility

    Returns
    -------
    (lower, upper, bootstrap_distribution)
    """
    rng = np.random.default_rng(random_state)
    n = len(data)
    boot_stats = np.array([
        statistic(rng.choice(data, size=n, replace=True))
        for _ in range(n_boot)
    ])
    alpha = 1 - confidence
    lower = np.percentile(boot_stats, 100 * alpha / 2)
    upper = np.percentile(boot_stats, 100 * (1 - alpha / 2))
    return lower, upper, boot_stats


def coverage_simulation(
    true_mean: float,
    true_std: float,
    n: int,
    n_simulations: int = 2000,
    confidence: float = 0.95
) -> dict:
    """
    Simulate coverage probability: what fraction of CIs contain the true mean?

    Ideal: fraction should equal the confidence level (e.g. 0.95 for 95% CI).
    This verifies the CI method is well-calibrated.

    Parameters
    ----------
    n              : sample size per simulation
    n_simulations  : number of independent simulations

    Returns
    -------
    dict with coverage rates for z-CI, t-CI, and bootstrap CI
    """
    rng = np.random.default_rng(42)
    z_contains, t_contains, boot_contains = 0, 0, 0

    for sim in range(n_simulations):
        sample = rng.normal(true_mean, true_std, size=n)

        z_lo, z_hi, _, _ = z_confidence_interval(sample, confidence)
        t_lo, t_hi, _, _ = t_confidence_interval(sample, confidence)
        b_lo, b_hi, _ = bootstrap_ci(sample, n_boot=1000, confidence=confidence,
                                      random_state=sim)

        z_contains += int(z_lo <= true_mean <= z_hi)
        t_contains += int(t_lo <= true_mean <= t_hi)
        boot_contains += int(b_lo <= true_mean <= b_hi)

    return {
        "z_coverage": z_contains / n_simulations,
        "t_coverage": t_contains / n_simulations,
        "boot_coverage": boot_contains / n_simulations,
        "target": confidence,
    }


# ------ Bootstrap demo ------
data = np.random.normal(50, 10, size=50)

b_lo, b_hi, boot_dist = bootstrap_ci(data, statistic=np.mean, n_boot=10_000)
t_lo, t_hi, _, _ = t_confidence_interval(data)

print("Bootstrap vs t-CI comparison (n=50, statistic=mean)")
print(f"  Bootstrap: [{b_lo:.2f}, {b_hi:.2f}]  width = {b_hi - b_lo:.2f}")
print(f"  t-CI:      [{t_lo:.2f}, {t_hi:.2f}]  width = {t_hi - t_lo:.2f}")
print(f"  Both should be similar for normal data with n=50")

# Bootstrap CI for the median (t-CI doesn't directly apply)
b_lo_med, b_hi_med, _ = bootstrap_ci(data, statistic=np.median, n_boot=10_000)
print(f"\nBootstrap CI for median: [{b_lo_med:.2f}, {b_hi_med:.2f}]")
print("  No parametric equivalent - bootstrap shines here")

# ------ Coverage probability simulation ------
print("\nCoverage probability simulation (n=20, 2000 simulations):")
cov = coverage_simulation(true_mean=50, true_std=10, n=20, n_simulations=2000)
for method, rate in cov.items():
    if method != "target":
        target = cov["target"]
        flag = "OK" if abs(rate - target) < 0.03 else "CHECK"
        print(f"  {method}: {rate:.3f}  (target={target:.3f})  [{flag}]")

print("\nCoverage for n=5 (very small sample - bootstrap may undercover):")
cov_small = coverage_simulation(true_mean=50, true_std=10, n=5, n_simulations=2000)
for method, rate in cov_small.items():
    if method != "target":
        print(f"  {method}: {rate:.3f}  (target={cov_small['target']:.3f})")

## Real-World Example 1: CI for Conversion Rate (Proportion)

When the metric is a proportion (e.g. click-through rate, conversion rate), we have two choices:
- **Normal approximation**: CI = p_hat +/- z * sqrt(p*(1-p)/n) -- simple but inaccurate for extreme proportions
- **Wilson score interval**: more accurate, especially when p is near 0 or 1, or n is small

Rule of thumb: use Wilson score when n*p < 10 or n*(1-p) < 10.

In [ ]:
# -----------------------------------------------------------------------
# Real-World Example 1: CI for conversion rate (proportion)
# -----------------------------------------------------------------------

def normal_proportion_ci(
    successes: int, n: int, confidence: float = 0.95
) -> Tuple[float, float]:
    """Normal approximation CI for a proportion."""
    p_hat = successes / n
    se = np.sqrt(p_hat * (1 - p_hat) / n)
    z_star = stats.norm.ppf(1 - (1 - confidence) / 2)
    moe = z_star * se
    lower = max(0.0, p_hat - moe)   # clip to [0, 1]
    upper = min(1.0, p_hat + moe)
    return lower, upper


def wilson_score_ci(
    successes: int, n: int, confidence: float = 0.95
) -> Tuple[float, float]:
    """
    Wilson score interval for a proportion.

    More accurate than normal approximation near 0 and 1.
    Used by major tech companies for conversion rate CIs.
    """
    p_hat = successes / n
    z_star = stats.norm.ppf(1 - (1 - confidence) / 2)
    z2 = z_star ** 2
    centre = (p_hat + z2 / (2 * n)) / (1 + z2 / n)
    margin = z_star * np.sqrt(
        (p_hat * (1 - p_hat) / n + z2 / (4 * n ** 2)) / (1 + z2 / n)
    )
    return max(0.0, centre - margin), min(1.0, centre + margin)


# Scenario: 2% conversion rate (low proportion, common in e-commerce)
scenarios = [
    (2, 100, "n=100, k=2 (2%)"),
    (10, 500, "n=500, k=10 (2%)"),
    (40, 2000, "n=2000, k=40 (2%)"),
    (100, 1000, "n=1000, k=100 (10%)"),
    (450, 500, "n=500, k=450 (90%)"),
]

print(f"{'Scenario':<30}  {'Normal CI':<24}  {'Wilson CI':<24}  Note")
print("-" * 100)
for k, n, label in scenarios:
    norm_lo, norm_hi = normal_proportion_ci(k, n)
    wils_lo, wils_hi = wilson_score_ci(k, n)
    # Wilson is more accurate; normal can go negative for low p
    note = "agree" if abs(norm_lo - wils_lo) < 0.005 else "DIFFER - use Wilson"
    print(f"{label:<30}  [{norm_lo:.4f}, {norm_hi:.4f}]  [{wils_lo:.4f}, {wils_hi:.4f}]  {note}")

# A/B testing scenario: landing page conversion
n_visitors = 1200
n_conversions = 36  # 3% conversion rate
lo, hi = wilson_score_ci(n_conversions, n_visitors)
print(f"\nLanding page: {n_conversions}/{n_visitors} conversions")
print(f"Wilson 95% CI: [{lo:.3%}, {hi:.3%}]")
print(f"Point estimate: {n_conversions/n_visitors:.3%}")
print(f"CI width: {hi - lo:.3%} -- interpretation: true rate is plausibly 2.2% to 4.2%")

## Real-World Example 2: CI for Difference in Means (A/B Testing)

In A/B testing, the key question is: "does treatment B outperform control A?"
We compute the CI for the **difference** mu_B - mu_A.
If the CI **excludes 0**, the difference is statistically significant at the chosen level.
If the CI **excludes the MDE threshold**, the difference is practically significant.

In [ ]:
# -----------------------------------------------------------------------
# Real-World Example 2: CI for difference in means in A/B test
# -----------------------------------------------------------------------

def ab_test_ci(
    control: np.ndarray,
    treatment: np.ndarray,
    confidence: float = 0.95,
    mde: float = 0.0
) -> dict:
    """
    Compute CI for the difference in means (treatment - control).

    Parameters
    ----------
    control    : control group outcomes
    treatment  : treatment group outcomes
    confidence : CI confidence level
    mde        : minimum detectable effect (practical significance threshold)

    Returns
    -------
    dict with CI bounds, significance, practical significance, effect size
    """
    n_c, n_t = len(control), len(treatment)
    mean_c, mean_t = np.mean(control), np.mean(treatment)
    var_c = np.var(control, ddof=1)
    var_t = np.var(treatment, ddof=1)

    diff = mean_t - mean_c
    se_diff = np.sqrt(var_c / n_c + var_t / n_t)

    # Welch's t degrees of freedom (handles unequal variance)
    df_num = (var_c / n_c + var_t / n_t) ** 2
    df_den = (var_c / n_c) ** 2 / (n_c - 1) + (var_t / n_t) ** 2 / (n_t - 1)
    df = df_num / df_den

    t_star = stats.t.ppf(1 - (1 - confidence) / 2, df=df)
    moe = t_star * se_diff
    lower, upper = diff - moe, diff + moe

    statistically_significant = lower > 0 or upper < 0  # CI excludes 0
    practically_significant = lower > mde or upper < -mde

    return {
        "diff": diff, "lower": lower, "upper": upper,
        "se": se_diff, "df": df,
        "statistically_significant": statistically_significant,
        "practically_significant": practically_significant,
    }


# Simulate A/B test: revenue per session (dollars)
np.random.seed(42)
control = np.random.lognormal(mean=3.0, sigma=0.8, size=500)   # baseline revenue
treatment = np.random.lognormal(mean=3.08, sigma=0.8, size=500) # ~8% lift

result = ab_test_ci(control, treatment, confidence=0.95, mde=1.0)

print("A/B Test: Revenue per Session")
print(f"  Control mean:   ${np.mean(control):.2f} (n={len(control)})")
print(f"  Treatment mean: ${np.mean(treatment):.2f} (n={len(treatment)})")
print(f"  Observed lift:  ${result['diff']:.2f}")
print(f"  95% CI for lift: [${result['lower']:.2f}, ${result['upper']:.2f}]")
print(f"  Statistically significant (CI excludes 0): {result['statistically_significant']}")
print(f"  Practically significant (CI excludes MDE=${1.00}): {result['practically_significant']}")

# Scenario 2: no real effect
treatment_null = np.random.lognormal(mean=3.0, sigma=0.8, size=500)
result_null = ab_test_ci(control, treatment_null, confidence=0.95)
print(f"\nNull scenario (no real lift):")
print(f"  95% CI: [${result_null['lower']:.2f}, ${result_null['upper']:.2f}]")
print(f"  Statistically significant: {result_null['statistically_significant']}")
print("  CI includes 0 -- correctly fails to detect a non-existent effect")

## Real-World Example 3 + Comparison: CI Width vs Sample Size

A fundamental property of CIs: **width shrinks as 1/sqrt(n)**.
To halve the CI width, you must quadruple the sample size.

We visualize this relationship and show how to compute the required n for a target margin of error.

In [ ]:
# -----------------------------------------------------------------------
# Real-World Example 3 + Comparison: CI width vs sample size
# -----------------------------------------------------------------------

def required_sample_size(
    sigma: float,
    target_moe: float,
    confidence: float = 0.95
) -> int:
    """
    Compute minimum n to achieve a target margin of error for the mean.

    n = ceil( (z * sigma / MOE)^2 )
    """
    z_star = stats.norm.ppf(1 - (1 - confidence) / 2)
    n = (z_star * sigma / target_moe) ** 2
    return int(np.ceil(n))


# ------ Plot CI width vs sample size ------
sigma = 10.0   # known population std
ns = np.arange(10, 1001, 10)
z_star = stats.norm.ppf(0.975)
widths_z = 2 * z_star * sigma / np.sqrt(ns)          # z-CI width
widths_t = np.array([2 * stats.t.ppf(0.975, df=n-1) * sigma / np.sqrt(n)
                     for n in ns])                    # t-CI width

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

ax1 = axes[0]
ax1.plot(ns, widths_z, color="steelblue", linewidth=2, label="z-CI (sigma known)")
ax1.plot(ns, widths_t, color="darkorange", linewidth=2, linestyle="--",
         label="t-CI (sigma estimated)")
ax1.axhline(y=4.0, color="red", linestyle=":", linewidth=1.5, label="Target width = 4.0")
n_req = required_sample_size(sigma, target_moe=2.0)
ax1.axvline(x=n_req, color="green", linestyle=":", linewidth=1.5,
            label=f"n required = {n_req}")
ax1.set_xlabel("Sample Size (n)", fontsize=12)
ax1.set_ylabel("95% CI Width", fontsize=12)
ax1.set_title("CI Width vs Sample Size (sigma=10)", fontsize=13, fontweight="bold")
ax1.legend(fontsize=9)
ax1.grid(alpha=0.3)

# ------ Visualize 3 CI methods for same data ------
sample = np.random.normal(50, 10, size=40)
z_lo, z_hi, mean_val, _ = z_confidence_interval(sample)
t_lo, t_hi, _, _ = t_confidence_interval(sample)
b_lo, b_hi, _ = bootstrap_ci(sample, n_boot=10_000)

ax2 = axes[1]
methods = ["z-CI
(sigma known)", "t-CI
(sigma estimated)", "Bootstrap CI
(distribution-free)"]
lowers = [z_lo, t_lo, b_lo]
uppers = [z_hi, t_hi, b_hi]
colors = ["steelblue", "darkorange", "seagreen"]

for i, (lo, hi, method, color) in enumerate(zip(lowers, uppers, methods, colors)):
    ax2.barh(i, hi - lo, left=lo, height=0.5, color=color, alpha=0.7, label=method)
    ax2.text((lo + hi) / 2, i, f"[{lo:.1f}, {hi:.1f}]",
             ha="center", va="center", fontsize=9, fontweight="bold")

ax2.axvline(x=50, color="black", linestyle="-", linewidth=2, label="True mean = 50")
ax2.axvline(x=mean_val, color="red", linestyle="--", linewidth=1.5, label=f"Sample mean = {mean_val:.1f}")
ax2.set_yticks(range(3))
ax2.set_yticklabels(methods, fontsize=10)
ax2.set_xlabel("Value", fontsize=12)
ax2.set_title("Three 95% CI Methods for Same Dataset (n=40)", fontsize=13, fontweight="bold")
ax2.legend(fontsize=9, loc="upper right")
ax2.grid(alpha=0.3, axis="x")

plt.tight_layout()
plt.savefig("ci_comparison.png", dpi=100, bbox_inches="tight")
plt.show()

# ------ Required sample size table ------
print("Required n for target margin of error (sigma=10, 95% CI):")
print(f"{'Target MOE':>12}  {'Required n':>12}  {'Relative to MOE=5':>20}")
n_base = required_sample_size(sigma=10, target_moe=5.0)
for moe in [5.0, 2.5, 2.0, 1.0, 0.5]:
    n_req = required_sample_size(sigma=10, target_moe=moe)
    ratio = n_req / n_base
    print(f"{moe:>12.1f}  {n_req:>12d}  {ratio:>18.1f}x")
print("Halving MOE quadruples required n (the 1/sqrt(n) relationship)")